<a href="https://colab.research.google.com/github/anandachaturvedi/Orchestrator_poc/blob/main/Real_Orchestrator_Dynamic_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U langgraph langchain-openai langchain

In [2]:
import os
import json
from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from google.colab import userdata

In [4]:
OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
MODEL              = "gpt-4o-mini"

llm = ChatOpenAI(
    model           = MODEL,
    api_key         = OPENROUTER_API_KEY,
    base_url        = "https://openrouter.ai/api/v1",
)

In [ ]:
class OrchestratorState(TypedDict):
    topic          : str
    agents_to_call : List[str]   # LLM fills this in node 1
    agent_outputs  : dict        # dynamic dict — stores ANY agent's output
    retry_agent    : str         # LLM fills this if it wants a retry
    retry_count    : int
    final          : str

In [ ]:
def llm_orchestrator(state: OrchestratorState) -> dict:

    prompt = f"""You are an orchestrator. A user wants to learn about: "{state['topic']}"

You have 3 agents available:
- "summarizer"  : writes a short summary of the topic
- "quiz"        : creates 5 quiz questions about the topic
- "use_cases"   : lists 3 real world use cases of the topic

Your Job:
Decide which agents are actually useful for this topic.
- If the topic is a concept (like AI, blockchain) → use all 3
- If the topic is a person (like Elon Musk) → skip quiz, use summarizer + use_cases
- If the topic is very narrow/technical → use summarizer + use_cases only

Respond ONLY in this exact JSON format, nothing else:
{{
  "agents_to_call": ["summarizer", "quiz", "use_cases"],
  "reason": "your reason here"
}}"""

    result = llm.invoke(prompt)

    try:
        raw = result.content.strip()
        if raw.startswith("```"):
            raw = raw.split("```")[1]
            if raw.startswith("json"):
                raw = raw[4:]
        decision       = json.loads(raw.strip())
        agents_to_call = decision.get("agents_to_call", ["summarizer", "use_cases"])
        reason         = decision.get("reason", "")
    except Exception:
        agents_to_call = ["summarizer", "use_cases"]
        reason         = "fallback decision"

    print(f"   LLM decided to call: {agents_to_call}")
    print(f"   Reason: {reason}\n")

    return {
        "agents_to_call": agents_to_call,
        "agent_outputs":  {},   # start with empty dict — agents will fill it
        "retry_agent":    "",
        "retry_count":    state.get("retry_count", 0),
    }

In [ ]:
def run_agents(state: OrchestratorState) -> dict:
    agents = state["agents_to_call"]
    topic = state["topic"]
    agent_outputs = {}

    for agent in agents:
        print(f"Agent - {agent} running")

        prompt = f"""
You are the "{agent}" agent.

Your job is to process the topic: "{topic}".

Please perform the task assigned to your role.
Your response should be:
- Accurate
- Detailed
- Beginner-friendly
- Well-structured
"""
        result = llm.invoke(prompt)
        agent_outputs[agent] = result.content

    return {
        "agent_outputs": agent_outputs
    }

In [ ]:
def llm_quality_check(state: OrchestratorState) -> dict:

    # don't retry more than once
    if state.get("retry_count", 0) >= 1:
        print("Already retried once. Accepting results as-is.\n")
        return {"retry_agent": ""}

    # build report from whatever agents ran
    outputs = []
    for agent, output in state.get("agent_outputs", {}).items():
        outputs.append(f"{agent.upper()} output (first 200 chars): {output[:200]}")

    outputs_text = "\n".join(outputs)

    prompt = f"""You are an orchestrator checking the quality of agent outputs for topic: "{state['topic']}"

Here are the outputs:
{outputs_text}

Check if any output is:
- Too short (under 50 words)
- Irrelevant to the topic
- Clearly wrong or empty

If everything looks good respond ONLY with:
{{"status": "ok", "retry_agent": ""}}

If one agent needs a retry respond ONLY with:
{{"status": "retry", "retry_agent": "agent_name"}}
(replace agent_name with the actual agent name that needs retry)

Respond ONLY in JSON. Nothing else."""

    result = llm.invoke(prompt)

    try:
        raw = result.content.strip()
        if raw.startswith("```"):
            raw = raw.split("```")[1]
            if raw.startswith("json"):
                raw = raw[4:]
        decision    = json.loads(raw.strip())
        retry_agent = decision.get("retry_agent", "")
        status      = decision.get("status", "ok")
    except Exception:
        retry_agent = ""
        status      = "ok"

    if status == "retry" and retry_agent:
        print(f"LLM says retry agent: {retry_agent}\n")
    else:
        print("LLM says all outputs look good\n")

    return {
        "retry_agent": retry_agent,
        "retry_count": state.get("retry_count", 0) + 1,
    }

In [ ]:
def retry_agent(state: OrchestratorState) -> dict:
    agent = state["retry_agent"]          # Agent selected by the LLM
    topic = state["topic"]
    agent_outputs = state.get("agent_outputs", {})

    print(f"Re-running agent: {agent}\n")

    # Prompt for retrying the failed agent
    prompt = f"""
The "{agent}" agent produced a poor-quality output for the topic "{topic}".

Please redo this task properly.

Requirements:
- Be detailed and accurate.
- Make the response beginner-friendly.
- Include relevant information.
- Ensure the output is at least 150 words.
"""
    result = llm.invoke(prompt)
    agent_outputs[agent] = result.content

    return {
        "agent_outputs": agent_outputs
    }

In [ ]:
def llm_combine(state: OrchestratorState) -> dict:
    print("LLM Orchestrator is combining all results into final answer...\n")

    agents        = state["agents_to_call"]
    agent_outputs = state.get("agent_outputs", {})

    # dynamically collect only agents that ran and have output
    sections = []
    for agent in agents:
        if agent_outputs.get(agent):   # only if this agent produced output
            sections.append(f"{agent.upper()}:\n{agent_outputs[agent]}")

    combined = "\n\n".join(sections)

    prompt = f"""You are an orchestrator. Here are the outputs from your agents about "{state['topic']}":

{combined}

Combine these into one clean, well-structured final report.
Add a one-line intro at the top.
Keep all the content, just format it nicely."""

    result = llm.invoke(prompt)

    final = f"""
{'='*60}
TOPIC: {state['topic'].upper()}
{'='*60}
{result.content}
{'='*60}
"""
    return {"final": final}

In [ ]:
def route_after_quality_check(state: OrchestratorState) -> str:
    if state.get("retry_agent"):
        return "retry_agent"   # LLM wants a retry
    return "llm_combine"       # LLM all good...combine

In [ ]:
def build_graph():
    graph = StateGraph(OrchestratorState)

    graph.add_node("llm_orchestrator",  llm_orchestrator)
    graph.add_node("run_agents",        run_agents)
    graph.add_node("llm_quality_check", llm_quality_check)
    graph.add_node("retry_agent",       retry_agent)
    graph.add_node("llm_combine",       llm_combine)

    graph.set_entry_point("llm_orchestrator")
    graph.add_edge("llm_orchestrator",  "run_agents")
    graph.add_edge("run_agents",        "llm_quality_check")
    graph.add_edge("retry_agent",       "llm_combine")
    graph.add_edge("llm_combine",       END)

    graph.add_conditional_edges(
        "llm_quality_check",
        route_after_quality_check,
        {
            "retry_agent": "retry_agent",
            "llm_combine": "llm_combine",
        }
    )

    return graph.compile()

In [ ]:
if __name__ == "__main__":
    topic = input("Enter a topic: ").strip()
    if not topic:
        topic = "Artificial Intelligence"

    print(f"Starting the orchestrator for: '{topic}'")
    print("LLM will decide which agents to call...\n")

    app    = build_graph()
    result = app.invoke({
        "topic":          topic,
        "agents_to_call": [],
        "agent_outputs":  {},    # single dynamic dict replaces summary/quiz/use_cases
        "retry_agent":    "",
        "retry_count":    0,
        "final":          "",
    })

    print(result["final"])

Enter a topic: sun
Starting the orchestrator for: 'sun'
LLM will decide which agents to call...

   LLM decided to call: ['summarizer', 'quiz', 'use_cases']
   Reason: The topic 'sun' encompasses various concepts including its physical characteristics, importance in solar energy, and its role in astronomy, making it suitable for a broad understanding that can benefit from a summary, quiz, and real-world use cases.

Agent - summarizer is running
Agent - quiz is running
Agent - use_cases is running
LLM says all outputs look good

LLM Orchestrator is combining all results into final answer...


TOPIC: SUN
# Comprehensive Report on the Sun

This report provides an in-depth analysis of the Sun, covering its structure, importance, use cases, and a quiz designed for educational purposes.

---

## Summary of the Sun

### Overview
The Sun is a massive star located at the center of our solar system. It is a nearly perfect sphere of hot plasma, predominantly composed of hydrogen (about 74%) and h